In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent

c:\Users\geetikak\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# llm
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

NameError: name 'ChatGroq' is not defined

Adding Checkpointer to remember the previous messages

In [3]:
@tool
def add(a: int, b: int):
    """Add two numbers."""
    return a + b



In [7]:
# list of exisitng tools
tools = [add]
llm_with_tools = llm.bind_tools(tools)

In [16]:
# Agent node
def call_model(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(
        state["messages"]
    )
    return {
        "messages": [response]
    }

In [17]:
# Router
def should_continue(state: MessagesState)-> str:
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "tools"
    return END

In [18]:
# Build Graph-
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools)) # without creating def tools, this node was created via  langgraph.prebuilt's ToolNode.

# Add edges
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {
    "tools": "tools",
    END: END
})
builder.add_edge("tools", "agent")



In [19]:
# Add checkointer

checkpointer = InMemorySaver()
graph = builder.compile(
    checkpointer = checkpointer
)

In [26]:
# Conversation Identity

config1 = {
    "configurable":{
        "thread_id": "converstion-1"
    }
}

In [31]:
config2 = {
    "configurable": {
    "thread_id": "conversation-2"
    }
}

In [32]:
# FIRST CALL

res1 = graph.invoke({
    "messages": [{
        "role": "user",
        "content": "my name is Sam "
}]
},
config1)

In [33]:
print(res1['messages'][-1].content)

Nice to meet you, Sam! If there's anything you'd like to chat about or need help with, just let me know.


In [34]:
# SECOND CALL
res2 = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what is my name?"
            }
        ]
    }, config1
)

In [35]:
print(res2['messages'][-1].content)

Your name is **Sam**.


In [38]:
res3 = graph.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content":"what is my name?"
            }
        ]
    }, config2
)

print(res3['messages'][-1].content)
print(res3)

I don’t have any record of your name. Could you let me know what you’d like to be called?
{'messages': [HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id='db263ad2-4c32-4230-8576-780c64afa8b2'), AIMessage(content='I don’t have any information about your name. Could you let me know what you’d like to be called?', additional_kwargs={'reasoning_content': 'The user asks "what is my name?" There\'s no prior context. We don\'t have user info. We cannot guess. We should respond politely asking for clarification or stating we don\'t know.'}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 125, 'total_tokens': 195, 'completion_time': 0.147227334, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.004682431, 'prompt_tokens_details': None, 'queue_time': 0.283969505, 'total_time': 0.151909765}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reaso